# Lab 12: LSTM for Time Series Forecasting

**University of Engineering and Technology Peshawar, Nowshera Campus**

**Course:** Machine Learning Lab

**Student Name:** Muhammad Ayub
**Registration Number:** 22jzele0470

**Date:** May 22, 2026

---

## Objective
Implement Long Short-Term Memory (LSTM) network for univariate multi-step time series forecasting on AEP hourly energy consumption dataset.

## Table of Contents
1. [Import Libraries](#imports)
2. [Model Definition](#model)
3. [Data Loading & Preprocessing](#data)
4. [Model Training](#training)
5. [Evaluation & Results](#evaluation)
6. [Conclusion](#conclusion)

## 1. Import Libraries <a id='imports'></a>

In [ ]:
import os
import time
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow
import tensorflow.keras.backend as K
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import LSTM, Dense, Flatten, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.callbacks.TrainingMonitor import TrainingMonitor

# Update this path according to your system
os.chdir(r'C:\Users\M Ayub\Downloads\ML_LAB')

## 2. Model Definition <a id='model'></a>

In [ ]:
time_steps = 24
num_features = 21

def create_lstm():
    """Define LSTM model architecture."""
    input_data = Input(shape=(time_steps, num_features))
    lstm1 = LSTM(8, return_sequences=True)(input_data)
    lstm2 = LSTM(20)(lstm1)
    x = Flatten()(lstm2)
    output = Dense(1)(x)
    model = Model(input_data, output)
    return model

model = create_lstm()
model.summary()

In [ ]:
# Visualize model (optional - requires pydot + graphviz)
try:
    tensorflow.keras.utils.plot_model(model, show_shapes=True, to_file='lstm_model.png')
    print("Model architecture saved as lstm_model.png")
except Exception as e:
    print("Could not generate model plot. Error:", e)

## 3. Data Loading & Preprocessing <a id='data'></a>

In [ ]:
# Load preprocessed data
path_dataset = r'C:\Users\Administrator\Downloads\ML Lab\AEP_hourly\processed'

train_set = pd.read_csv(os.path.join(path_dataset, 'AEP_train.csv')).values
validation_set = pd.read_csv(os.path.join(path_dataset, 'AEP_validation.csv')).values
test_set = pd.read_csv(os.path.join(path_dataset, 'AEP_test.csv')).values

scaler = pickle.load(open(os.path.join(path_dataset, 'AEP_Scaler.pkl'), 'rb'))

print("Train shape:", train_set.shape)
print("Validation shape:", validation_set.shape)
print("Test shape:", test_set.shape)

In [ ]:
# Create time series sequences
start = time.time()
train_X, train_y = univariate_multi_step(train_set, time_steps, target_col=0, target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0, target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0, target_len=1)
print(f"Data preparation time: {time.time() - start:.2f} seconds")

## 4. Model Training <a id='training'></a>

In [ ]:
# Create output directory
OUTPUT_PATH = r'C:\Users\Administrator\Downloads\ML Lab\checkpoint\ML Lab\lab12'
os.makedirs(OUTPUT_PATH, exist_ok=True)

checkpoints = os.path.join(OUTPUT_PATH, 'E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5')
FIG_PATH = os.path.join(OUTPUT_PATH, 'history.png')
JSON_PATH = os.path.join(OUTPUT_PATH, 'history.json')

checkpoint_callback = ModelCheckpoint(checkpoints, monitor="val_loss", save_best_only=True, verbose=1)
monitor_callback = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=0)
callbacks = [checkpoint_callback, monitor_callback]

In [ ]:
# Compile the model
model.compile(loss='mae', optimizer=Adam(learning_rate=1e-3), metrics=["mae", "mape"])
print("[INFO] Model compiled.")

In [ ]:
# Train
epochs = 60
batch_size = 32

history = model.fit(
    train_X, train_y,
    validation_data=(validation_X, validation_y),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1
)

## 5. Evaluation & Results <a id='evaluation'></a>

In [ ]:
# Load best model (update filename as per your best checkpoint)
best_model_path = os.path.join(OUTPUT_PATH, 'E1-cp-0033-loss0.01.h5')  # Change according to your best epoch
model = load_model(best_model_path)

# Make predictions
y_pred_scaled = model.predict(test_X)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)

# Calculate metrics
mae = np.mean(np.abs(y_pred - y_test_unscaled))
medae = np.median(np.abs(y_pred - y_test_unscaled))
mse = np.mean(np.square(y_pred - y_test_unscaled))
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
mdape = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100

print("=== Model Performance on Test Set ===")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Median Absolute Error: {medae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
print(f"Median Absolute Percentage Error (MDAPE): {mdape:.2f}%")

## 6. Conclusion <a id='conclusion'></a>

The LSTM model achieved good forecasting performance on the AEP hourly dataset. Key strengths include its ability to capture temporal dependencies. 

**Suggestions for Improvement:**
- Hyperparameter tuning
- Use of Bidirectional LSTM or GRU
- Adding attention mechanism

**GitHub Link:** (https://github.com/prince4775/8th-Semester-ML-and-DL-Lab)

---
This notebook follows the lab report guidelines: clean structure, proper headings, code + outputs, and explanations.